# Session 22 — Explainable AI Pipeline with SHAP and Model Monitoring

**Goal:** go beyond "the model said 73% risk" to "the model said 73% risk *because
of these specific features*" — using [SHAP](https://shap.readthedocs.io/) (SHapley
Additive exPlanations) for per-prediction explainability, combined with monitoring
how those explanations shift over time.

## Why SHAP over the feature-importance from Session 5

Session 5's `feature_importances_` gives one global ranking ("age matters most,
overall"). SHAP gives a **per-prediction** breakdown ("for *this specific* patient,
age pushed risk up by 0.12 and cholesterol pushed it down by 0.03") — the difference
between "this model generally cares about X" and "this specific decision was driven
by X", which is what a clinician, loan officer, or regulator actually needs.

## Prerequisites

```bash
pip install shap
```
Runs entirely locally.

In [ ]:
import pandas as pd
import numpy as np
import shap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

print("shap version:", shap.__version__)

## Step 1 — Train a model on the heart disease dataset

Same dataset as Session 15, so the explanations below are directly comparable to
that session's aggregate metrics.

In [ ]:
df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
X, y = df.drop(columns="target"), df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=0)
model.fit(X_train, y_train)
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"test AUC: {auc:.4f}")

## Step 2 — Compute SHAP values

`TreeExplainer` is an efficient, exact SHAP implementation for tree-based models
(random forests, gradient boosting) — for other model types, `shap.Explainer` picks
an appropriate approximation automatically.

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

print(f"SHAP values shape: {shap_values.values.shape}")  # (n_samples, n_features, n_classes)
print("Base value (average model output over training data):", explainer.expected_value)

## Step 3 — Global explanation: which features matter most, on average

This is SHAP's aggregate view — comparable to Session 5's `feature_importances_`,
but grounded in the same per-prediction values used for individual explanations
below, so the two views are consistent by construction.

In [ ]:
mean_abs_shap = pd.Series(
    np.abs(shap_values.values[:, :, 1]).mean(axis=0), index=X.columns
).sort_values(ascending=False)
print("Mean |SHAP value| per feature (class=disease present):")
print(mean_abs_shap.round(4))

## Step 4 — Per-prediction explanation

The payoff: explain one specific patient's prediction in terms of which features
pushed the risk score up or down, and by how much.

In [ ]:
def explain_prediction(idx):
    row = X_test.iloc[idx]
    proba = model.predict_proba(row.values.reshape(1, -1))[0, 1]
    contributions = pd.Series(shap_values.values[idx, :, 1], index=X.columns).sort_values(key=abs, ascending=False)

    print(f"Patient {idx}: predicted risk = {proba:.3f}")
    print(f"Base rate (average patient): {explainer.expected_value[1]:.3f}")
    print("\nTop feature contributions:")
    for feature, contribution in contributions.head(6).items():
        direction = "increases" if contribution > 0 else "decreases"
        print(f"  {feature:<12} = {row[feature]:<8}  {direction} risk by {abs(contribution):.4f}")

explain_prediction(0)

In [ ]:
print("--- A different patient, for contrast ---\n")
explain_prediction(5)

## Step 5 — Wire explanations into the FastAPI serving layer

A real explainable-AI deployment returns the explanation *alongside* the prediction,
not as a separate offline report — following the same API shape as Session 7 and
Session 15.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI(title="Explainable Heart Disease Risk API")

class PatientFeatures(BaseModel):
    age: float; sex: float; cp: float; trestbps: float; chol: float
    fbs: float; restecg: float; thalach: float; exang: float
    oldpeak: float; slope: float; ca: float; thal: float

@app.post("/v1/predict-with-explanation")
def predict_with_explanation(features: PatientFeatures):
    row = np.array([[getattr(features, c) for c in X.columns]])
    proba = model.predict_proba(row)[0, 1]
    row_shap = explainer(row)
    contributions = {
        col: round(float(row_shap.values[0, i, 1]), 4) for i, col in enumerate(X.columns)
    }
    top_drivers = sorted(contributions.items(), key=lambda kv: abs(kv[1]), reverse=True)[:5]

    return {
        "risk_probability": round(float(proba), 4),
        "top_drivers": [{"feature": f, "contribution": c} for f, c in top_drivers],
    }

client = TestClient(app)
sample = X_test.iloc[0].to_dict()
response = client.post("/v1/predict-with-explanation", json=sample)
print(response.status_code)
print(response.json())

## Step 6 — Monitor whether *explanations* drift, not just predictions

A subtler failure mode than the data drift from Session 5: the model might keep the
same accuracy while *shifting which features it relies on* — worth tracking
separately, since a clinician trusting "age matters most" would want to know if that
stopped being true.

In [ ]:
def mean_abs_shap_ranking(shap_vals, columns):
    return pd.Series(np.abs(shap_vals[:, :, 1]).mean(axis=0), index=columns).sort_values(ascending=False)

reference_ranking = mean_abs_shap_ranking(shap_values.values, X.columns)

# Simulate a later time window
simulated_later_values = shap_values.values.copy()
simulated_later_values[:, list(X.columns).index("chol"), 1] *= 2.5  # cholesterol becomes far more influential
later_ranking = mean_abs_shap_ranking(simulated_later_values, X.columns)

comparison = pd.DataFrame({"reference_rank": reference_ranking.rank(ascending=False),
                            "later_rank": later_ranking.rank(ascending=False)})
comparison["rank_shift"] = comparison["reference_rank"] - comparison["later_rank"]
print(comparison.sort_values("rank_shift", key=abs, ascending=False))

## What to try next

* Use `shap.plots.beeswarm(shap_values)` (needs a plotting backend) for the classic
  SHAP summary plot — one dot per patient per feature, colored by feature value.
* Combine with Session 5's Evidently drift report: track both *what the data looks
  like* and *what the model relies on*, since either can shift independently.
* For the CNN from Session 20, `shap.DeepExplainer` (or the newer unified
  `shap.Explainer`) gives pixel-level attribution maps — the imaging analogue of
  this session's per-feature tabular attributions.